# GCP VM Complete End-to-End Setup
## AI 3D Reconstruction with Hunyuan Worker

Machine image đã có NVIDIA driver sẵn. Notebook này setup từ đầu đến cuối:
1. Clone repo
2. Setup Python environment
3. Install dependencies
4. Setup Hunyuan worker service
5. Setup FastAPI backend service
6. Verify all services
7. Setup Cloudflare tunnel

**Thời gian**: ~1-2 hours (lần đầu)

**Chú ý**: Chạy từng cell, đợi mỗi cell hoàn thành trước khi chạy cell tiếp theo

## Step 1: Verify NVIDIA GPU

Kiểm tra NVIDIA driver đã cài đặt và GPU available

In [ ]:
!nvidia-smi

Expected output: Show L4 or T4 GPU info. Nếu error → NVIDIA driver chưa cài đặt.

## Step 2: Install System Dependencies

In [ ]:
%%bash
set -euo pipefail

echo "=== Updating system packages ==="
sudo apt-get update
sudo apt-get upgrade -y
sudo apt-get install -y build-essential wget curl git tmux

echo "=== System packages installed ==="

## Step 3: Setup Python Environment

Tạo working directory và Python venv

In [ ]:
%%bash
set -euo pipefail

# Create working directory
mkdir -p ~/work
cd ~/work

# Clone repository
if [ ! -d "AI_3D_Reconstruction_Systerm" ]; then
  echo "Cloning repository..."
  git clone https://github.com/TangDien02/AI_3D_Reconstruction_Systerm.git
  cd AI_3D_Reconstruction_Systerm
else
  echo "Repository already exists, pulling latest..."
  cd AI_3D_Reconstruction_Systerm
  git pull origin codex/hunyuan-shape-then-paint
fi

# Create Python venv
echo "Creating Python virtual environment..."
python3.11 -m venv .venv
source .venv/bin/activate
python -m pip install -U pip wheel "setuptools<82"

echo "Python environment ready!"
python --version

## Step 4: Install Python Dependencies

⚠️ **WARNING**: Cài đặt này mất 15-30 phút (PyTorch + models rất nặng)

In [ ]:
%%bash
set -euo pipefail

cd ~/work/AI_3D_Reconstruction_Systerm
source .venv/bin/activate

echo "Installing Python dependencies... (this may take 15-30 minutes)"
export MPLBACKEND=Agg
python -m pip install --no-cache-dir -r requirements.txt

echo "Dependencies installed!"

## Step 5: Verify Runtime

Kiểm tra tất cả imports đã OK

In [ ]:
%%bash
set -euo pipefail

cd ~/work/AI_3D_Reconstruction_Systerm
source .venv/bin/activate

export MPLBACKEND=Agg
python scripts/verify_runtime.py

## Step 6: Download Hunyuan Model

⚠️ **WARNING**: Lần đầu download model ~50GB, mất 10-20 phút

Lần sau sẽ dùng cached model, nhanh hơn

In [ ]:
%%bash
set -euo pipefail

cd ~/work/AI_3D_Reconstruction_Systerm
source .venv/bin/activate

export MPLBACKEND=Agg
echo "Downloading Hunyuan model... (first time only, ~50GB)"

python -c "
import torch
from diffusers import DiffusionPipeline
print('Device:', 'CUDA' if torch.cuda.is_available() else 'CPU')
model_id = 'tencent/Hunyuan3D-2'
print(f'Loading {model_id}...')
pipeline = DiffusionPipeline.from_pretrained(
    model_id,
    subfolder='hunyuan3d-dit-v2-0',
    torch_dtype=torch.float16,
    device_map='auto'
)
print('✓ Model loaded successfully!')
"

echo "Model ready!"

## Step 7: Setup Hunyuan Worker Service

Tạo systemd service chạy Hunyuan worker ở port 8010

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="$HOME/work/AI_3D_Reconstruction_Systerm"
VENV_DIR="$REPO_DIR/.venv"

echo "Creating worker.env..."
cat > "$REPO_DIR/worker.env" <<'EOF'
MPLBACKEND=Agg
HUNYUAN_ENABLE_SHAPE=true
HUNYUAN_ENABLE_TEXTURE=false
EOF

echo "Creating systemd service for Hunyuan worker..."
sudo tee /etc/systemd/system/hunyuan-worker.service >/dev/null <<EOF
[Unit]
Description=Hunyuan 3D Worker Service
After=network-online.target
Wants=network-online.target

[Service]
Type=simple
User=$USER
WorkingDirectory=$REPO_DIR
EnvironmentFile=$REPO_DIR/worker.env
ExecStart=$VENV_DIR/bin/python -m uvicorn server.hunyuan_worker:app --host 0.0.0.0 --port 8010
Restart=on-failure
RestartSec=10

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl daemon-reload
sudo systemctl enable hunyuan-worker
sudo systemctl start hunyuan-worker

echo "Hunyuan worker service created!"
echo "Waiting 10s for service to start..."
sleep 10

## Step 8: Verify Hunyuan Worker

Kiểm tra worker đã ready

In [ ]:
%%bash
set -euo pipefail

echo "=== Hunyuan Worker Status ==="
sudo systemctl status hunyuan-worker --no-pager | head -20

echo ""
echo "=== Checking health endpoint ==="
for attempt in $(seq 1 30); do
  body=$(curl -fsS --max-time 5 http://127.0.0.1:8010/health 2>/dev/null || true)
  if [ -n "$body" ] && echo "$body" | python3 -m json.tool; then
    echo "✓ Hunyuan worker health OK after ${attempt}s"
    exit 0
  fi
  echo "Attempt $attempt/30..."
  sleep 1
done

echo "✗ Worker did not respond. Checking logs:"
sudo journalctl -u hunyuan-worker -n 50 --no-pager
exit 1

## Step 9: Setup FastAPI Backend Service

Tạo backend service ở port 8000, gọi worker qua 127.0.0.1:8010

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="$HOME/work/AI_3D_Reconstruction_Systerm"
VENV_DIR="$REPO_DIR/.venv"

echo "Creating backend.env..."
cat > "$REPO_DIR/backend.env" <<'EOF'
RECONSTRUCTION_BACKEND=hunyuan_remote
HUNYUAN_REMOTE_URL=http://127.0.0.1:8010
HUNYUAN_REMOTE_OUTPUT_FORMAT=glb
HUNYUAN_REMOTE_ENABLE_TEXTURE=false
HUNYUAN_REMOTE_TIMEOUT_SECONDS=1800
HUNYUAN_REMOTE_POLL_INTERVAL_SECONDS=5
IMAGE_CLEANER_BACKEND=auto
ENABLE_REMBG_CLEANER=true
CLEAN_IMAGE_MAX_SIDE=1536
CLEAN_IMAGE_PAD_RATIO=0.08
EOF

echo "Creating systemd service for FastAPI backend..."
sudo tee /etc/systemd/system/ai-3d-backend.service >/dev/null <<EOF
[Unit]
Description=AI 3D Reconstruction FastAPI backend
After=network-online.target
Wants=network-online.target

[Service]
Type=simple
User=$USER
WorkingDirectory=$REPO_DIR
EnvironmentFile=$REPO_DIR/backend.env
ExecStart=$VENV_DIR/bin/python -m uvicorn server.main:app --host 0.0.0.0 --port 8000
Restart=on-failure
RestartSec=10

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl daemon-reload
sudo systemctl enable ai-3d-backend
sudo systemctl start ai-3d-backend

echo "FastAPI backend service created!"
echo "Waiting 10s for service to start..."
sleep 10

## Step 10: Verify FastAPI Backend

Kiểm tra backend đã ready

In [ ]:
%%bash
set -euo pipefail

echo "=== FastAPI Backend Status ==="
sudo systemctl status ai-3d-backend --no-pager | head -20

echo ""
echo "=== Checking health endpoint ==="
for attempt in $(seq 1 30); do
  body=$(curl -fsS --max-time 5 http://127.0.0.1:8000/health 2>/dev/null || true)
  if [ -n "$body" ] && echo "$body" | python3 -m json.tool; then
    echo "✓ Backend health OK after ${attempt}s"
    exit 0
  fi
  echo "Attempt $attempt/30..."
  sleep 1
done

echo "✗ Backend did not respond. Checking logs:"
sudo journalctl -u ai-3d-backend -n 50 --no-pager
exit 1

## Step 11: Setup Cloudflare Tunnel

Tạo tunnel để expose backend ra ngoài internet (cần cho Expo app)

In [ ]:
%%bash
set -euo pipefail

# Download cloudflared
if [ ! -f "$HOME/.local/bin/cloudflared" ]; then
  echo "Downloading cloudflared..."
  mkdir -p "$HOME/.local/bin"
  wget -qO "$HOME/.local/bin/cloudflared" \
    https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x "$HOME/.local/bin/cloudflared"
  echo "✓ cloudflared downloaded"
else
  echo "✓ cloudflared already exists"
fi

# Create tunnel script
cat > "$HOME/work/start_tunnel.sh" <<'EOF'
#!/bin/bash
set -euo pipefail

echo "Starting Cloudflare tunnel on port 8000..."
echo "Press Ctrl+C to stop"
echo ""

$HOME/.local/bin/cloudflared tunnel --url http://127.0.0.1:8000 --no-autoupdate
EOF

chmod +x "$HOME/work/start_tunnel.sh"
echo "Tunnel script ready at ~/work/start_tunnel.sh"

## Step 12: Instructions for Running Tunnel

Chạy tunnel trong một SSH session khác (keep it running)

In [ ]:
print("""
╔════════════════════════════════════════════════════════════════╗
║          SETUP COMPLETE! Next steps:                           ║
╚════════════════════════════════════════════════════════════════╝

1️⃣  OPEN NEW SSH SESSION for Cloudflare tunnel:
    
    gcloud compute ssh admin@endtoend-gpu-dev --zone=asia-east1-b
    
    Then run:
    
    ~/work/start_tunnel.sh
    
    Copy the URL: https://RANDOM_NAME.trycloudflare.com

2️⃣  TEST Backend locally from THIS VM:
    
    curl http://127.0.0.1:8000/health
    
3️⃣  Setup Windows Expo app:
    
    # PowerShell
    cd C:\Users\YourUsername\Desktop\AI_3D_Reconstruction_Systerm_TangDien02\mobile
    
    # Using LAN IP (if on same network)
    $env:EXPO_PUBLIC_API_BASE_URL="http://192.168.1.6:8000"
    npm start -- --host lan
    
    # OR using Cloudflare tunnel URL
    $env:EXPO_PUBLIC_API_BASE_URL="https://RANDOM_NAME.trycloudflare.com"
    npm start -- --host lan

4️⃣  Verify services status:
    
    sudo systemctl status hunyuan-worker
    sudo systemctl status ai-3d-backend
    
5️⃣  View logs (if errors):
    
    sudo journalctl -u hunyuan-worker -f
    sudo journalctl -u ai-3d-backend -f
    
6️⃣  Restart services if needed:
    
    sudo systemctl restart hunyuan-worker
    sudo systemctl restart ai-3d-backend

╔════════════════════════════════════════════════════════════════╗
║  🎉 Setup complete! VM is ready for development               ║
╚════════════════════════════════════════════════════════════════╝
""")

## Step 13: Test Backend Preprocessing (Optional)

Test local preprocessing endpoint

In [ ]:
%%bash
# Test if sample image exists
TEST_IMAGE="$HOME/work/AI_3D_Reconstruction_Systerm/project/samples/chair_demo.png"

if [ -f "$TEST_IMAGE" ]; then
  echo "Testing preprocess endpoint with sample image..."
  curl -s -X POST "http://127.0.0.1:8000/preprocess/clean-image" \
    -F "image=@$TEST_IMAGE" \
    -F "bbox_x=10" \
    -F "bbox_y=10" \
    -F "bbox_width=400" \
    -F "bbox_height=400" \
    -F "job_id=test-preprocess" | python3 -m json.tool | head -50
else
  echo "Sample image not found. Skipping test."
fi

## Step 14: Summary

**Setup hoàn tất! ✅**

### Services đang chạy:
- ✅ Hunyuan Worker (port 8010)
- ✅ FastAPI Backend (port 8000)

### Next steps:
1. Chạy Cloudflare tunnel trong SSH session khác
2. Setup Windows Expo app
3. Test end-to-end reconstruction

### Useful commands:
```bash
# View worker logs
sudo journalctl -u hunyuan-worker -f

# View backend logs
sudo journalctl -u ai-3d-backend -f

# Restart worker
sudo systemctl restart hunyuan-worker

# Restart backend
sudo systemctl restart ai-3d-backend

# Stop both services
sudo systemctl stop hunyuan-worker ai-3d-backend

# Check GPU memory
nvidia-smi
```